# NSFnets 案例 1：Kovasznay 流（2D 定常）

**基于物理信息神经网络（PINN）求解不可压 Navier-Stokes 方程**

- 流动类型：Kovasznay 流（有解析解，可验证精度）
- 维度：2D 定常 (x, y)，不含时间项
- 雷诺数：Re = 40
- 网络结构：[2, 50, 50, 50, 50, 3] — 4 个隐藏层，每层 50 个神经元
- 损失函数：L_boundary（边界条件）+ L_residual（PDE 残差），无需初始条件

原 TF1 代码 by Zihao Hu (2020)，迁移至 MindSpore，添加中文注释。

In [1]:
import sys
sys.path.insert(0, '.')

import numpy as np
import matplotlib.pyplot as plt
import time

from nsfnet_module import NSFnet2D, relative_l2_error

# Reproducibility
np.random.seed(1234)

print('Imports complete.')

[WARNING] ME(2136:8589690880,MainProcess):2026-06-10-10:54:40.654.000 [mindspore/context.py:1338] For 'context.set_context', the parameter 'device_target' will be deprecated and removed in a future version. Please use the api mindspore.set_device() instead.


Imports complete.


## 1. 生成训练数据

Kovasznay 流解析解：
$$u = 1 - e^{\lambda x} \cos(2\pi y), \quad v = \frac{\lambda}{2\pi} e^{\lambda x} \sin(2\pi y), \quad p = \frac{1}{2}(1 - e^{2\lambda x})$$
其中 $\lambda = \frac{Re}{2} - \sqrt{\frac{Re^2}{4} + 4\pi^2}$，边界条件直接从解析解采样

In [2]:
# Domain and parameters
Re = 40
lam = 0.5 * Re - np.sqrt(0.25 * (Re ** 2) + 4 * (np.pi ** 2))

# Boundary points (Kovasznay: boundaries at x=-0.5, x=1.0, y=-0.5, y=1.5)
x_domain = np.linspace(-0.5, 1.0, 101)
y_domain = np.linspace(-0.5, 1.5, 101)

yb1 = np.array([-0.5] * 100)
yb2 = np.array([1.5] * 100)  # Note: original uses 1.0 but domain is [-0.5,1.5]
xb1 = np.array([-0.5] * 100)
xb2 = np.array([1.0] * 100)

# Boundary collocation points
y_train1 = np.concatenate([y_domain[1:101], y_domain[0:100], xb1, xb2], 0)
x_train1 = np.concatenate([yb1, yb2, x_domain[0:100], x_domain[1:101]], 0)

xb_train = x_train1.reshape(-1, 1).astype(np.float32)
yb_train = y_train1.reshape(-1, 1).astype(np.float32)

# Analytic solution on boundaries
ub_train = (1 - np.exp(lam * xb_train) * np.cos(2 * np.pi * yb_train)).astype(np.float32)
vb_train = (lam / (2 * np.pi) * np.exp(lam * xb_train) * np.sin(2 * np.pi * yb_train)).astype(np.float32)

# Interior collocation points (random, uniform)
N_train = 2601
x_train = ((np.random.rand(N_train, 1) - 1/3) * 3/2).astype(np.float32)
y_train = ((np.random.rand(N_train, 1) - 1/4) * 2).astype(np.float32)

print(f'Boundary points: {xb_train.shape[0]}')
print(f'Interior points: {x_train.shape[0]}')
print(f'lambda = {lam:.4f}')

Boundary points: 400
Interior points: 2601
lambda = -0.9637


## 2. 构建模型

网络架构：输入 (x,y) → 50 → 50 → 50 → 50 → 输出 (u, v, p)
输出分别为 x 方向速度、y 方向速度、压力

In [3]:
layers = [2, 50, 50, 50, 50, 3]

model = NSFnet2D(
    xb_train, yb_train, ub_train, vb_train,
    x_train, y_train, layers,
    Re=40.0, alpha=1.0
)

total_params = sum(int(np.prod(p.shape)) for p in model.net.trainable_params())
print(f'Total trainable parameters: {total_params}')

Total trainable parameters: 7953


## 3. 训练

**训练策略**（与原 TF1 代码一致）：
1. Adam 学习率=1e-3 × 5,000 步
2. Adam 学习率=1e-4 × 5,000 步
3. Adam 学习率=1e-5 × 50,000 步
4. Adam 学习率=1e-6 × 50,000 步
5. L-BFGS 二阶优化精调

Adam 总计 110,000 步，采用渐进降低学习率策略：先大学习率快速收敛，再逐步减小做精细调整

In [ ]:
print('=== Phase 1: Adam LR=1e-3 ===')
h1 = model.adam_train(nIter=5000, learning_rate=1e-3, print_every=500)

In [ ]:
print('=== Phase 2: Adam LR=1e-4 ===')
h2 = model.adam_train(nIter=5000, learning_rate=1e-4, print_every=500)

In [ ]:
print('=== Phase 3: Adam LR=1e-5 ===')
h3 = model.adam_train(nIter=50000, learning_rate=1e-5, print_every=5000)

In [ ]:
print('=== Phase 4: Adam LR=1e-6 ===')
h4 = model.adam_train(nIter=50000, learning_rate=1e-6, print_every=5000)

In [ ]:
print('=== Phase 5: L-BFGS Fine-tuning ===')
# LBFGS can be slow for large models. Reduce maxiter if needed.
lbfgs_result = model.lbfgs_train(maxiter=50000)

final_loss = float(model.loss_fn().asnumpy())
print(f'Final loss: {final_loss:.3e}')

## 4. 模型评估

在随机测试点上对比网络预测值与 Kovasznay 解析解，计算相对 L2 误差

In [ ]:
# Generate test data
np.random.seed(1234)
N_test = 1000

x_star = ((np.random.rand(N_test, 1) - 1/3) * 3/2).astype(np.float32)
y_star = ((np.random.rand(N_test, 1) - 1/4) * 2).astype(np.float32)

# Analytic solution
u_star = (1 - np.exp(lam * x_star) * np.cos(2 * np.pi * y_star)).astype(np.float32)
v_star = (lam / (2 * np.pi) * np.exp(lam * x_star) * np.sin(2 * np.pi * y_star)).astype(np.float32)
p_star = (0.5 * (1 - np.exp(2 * lam * x_star))).astype(np.float32)

# Prediction
t0 = time.time()
u_pred, v_pred, p_pred = model.predict(x_star, y_star)
print(f'Prediction time: {time.time() - t0:.2f}s')

In [ ]:
# Relative L2 errors
error_u = relative_l2_error(u_pred, u_star)
error_v = relative_l2_error(v_pred, v_star)
error_p = relative_l2_error(p_pred, p_star)

print(f'Relative L2 Error — u: {error_u:.3e}')
print(f'Relative L2 Error — v: {error_v:.3e}')
print(f'Relative L2 Error — p: {error_p:.3e}')

print()
print('=== Expected performance (original TF1 paper) ===')
print('Error u: ~1e-4 — 1e-3')
print('Error v: ~1e-4 — 1e-3')
print('Error p: ~1e-4 — 1e-3')

## 5. 可视化

在规则网格上绘制预测速度幅值、真实速度幅值、以及绝对误差的等高线图

In [ ]:
# Generate a fine grid for visualization
nx, ny = 100, 100
xg = np.linspace(-0.5, 1.0, nx, dtype=np.float32).reshape(-1, 1)
yg = np.linspace(-0.5, 1.5, ny, dtype=np.float32).reshape(-1, 1)

Xg, Yg = np.meshgrid(xg.ravel(), yg.ravel())
Xf = Xg.flatten().reshape(-1, 1).astype(np.float32)
Yf = Yg.flatten().reshape(-1, 1).astype(np.float32)

# Predict on grid
u_grid, v_grid, p_grid = model.predict(Xf, Yf)
U_mag_pred = np.sqrt(u_grid**2 + v_grid**2).reshape(ny, nx)

# Analytic on grid
u_true = 1 - np.exp(lam * Xf) * np.cos(2 * np.pi * Yf)
v_true = (lam / (2 * np.pi)) * np.exp(lam * Xf) * np.sin(2 * np.pi * Yf)
U_mag_true = np.sqrt(u_true**2 + v_true**2).reshape(ny, nx)

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

c0 = axes[0].contourf(Xg, Yg, U_mag_pred, levels=50, cmap='jet')
axes[0].set_title('Predicted |U|')
axes[0].set_xlabel('x'); axes[0].set_ylabel('y')
plt.colorbar(c0, ax=axes[0])

c1 = axes[1].contourf(Xg, Yg, U_mag_true, levels=50, cmap='jet')
axes[1].set_title('True |U|')
axes[1].set_xlabel('x'); axes[1].set_ylabel('y')
plt.colorbar(c1, ax=axes[1])

c2 = axes[2].contourf(Xg, Yg, np.abs(U_mag_pred - U_mag_true), levels=50, cmap='hot')
axes[2].set_title('Absolute Error')
axes[2].set_xlabel('x'); axes[2].set_ylabel('y')
plt.colorbar(c2, ax=axes[2])

plt.tight_layout()
plt.savefig('kovasznay_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: kovasznay_results.png')

## 总结

本 Notebook 演示了 2D 定常 Kovasznay 流的 MindSpore PINN 实现：

- **物理约束损失**：将边界数据拟合与 PDE 残差最小化相结合
- **自动微分**：通过 MindSpore 的 `ms.grad` 计算一阶和二阶空间偏导数
- **两阶段优化**：Adam（一阶随机梯度）→ L-BFGS（二阶确定性精调）
- **无需外部数据**：边界条件直接从 Kovasznay 解析解采样